In [1]:
import sys, os; sys.path.append(os.path.join(os.path.dirname("2_dataset_siamese.ipynb"), '..'))
from siamese.dataset.data_loader import data_loader

In [2]:
full_difficult = True

config_plds = {
    "train": [
        "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PKLot",
        "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/CNRParkEXT"
    ],
    "test": "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PLds",
    "split": 0.7,
    "full_difficult": full_difficult,
    "img_size": [128, 128]
}

config_cnrpark = {
    "train": [
        "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PKLot",
        "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PLds"
    ],
    "test": "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/CNRParkEXT",
    "split": 0.7,
    "full_difficult": full_difficult,
    "img_size": [128, 128]
}

config_pklot = {
    "train": [
        "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PLds",
        "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/CNRParkEXT"
    ],
    "test": "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PKLot",
    "split": 0.7,
    "full_difficult": full_difficult,
    "img_size": [128, 128]
}

In [3]:
config = config_pklot
ds_loader = data_loader(config)

train_spots = ds_loader.sets["train_spots"]
val_spots = ds_loader.sets["val_spots"]
train_cars = ds_loader.sets["train_cars"]
val_cars = ds_loader.sets["val_cars"]
test_sets = ds_loader.sets["test"]

##### Dataset read validation

In [4]:
print(f"Train and Val: {config['train'][0].split('/')[-1]}_{config['train'][1].split('/')[-1]}")

cars_in_train_spots = [car for cars in train_spots.values() for car in cars]
cars_in_val_spots = [car for cars in val_spots.values() for car in cars]

print("Cars qty:")
print(f"Train # By cars: {len(train_cars.keys())} - By spots: {len(cars_in_train_spots)}")
print(f"Val # By cars: {len(val_cars.keys())} - By spots: {len(cars_in_val_spots)}")

print("\nDifference:")
print(f"Train: {set(train_cars.keys()).difference(set(cars_in_train_spots))}")
print(f"Val: {set(val_cars.keys()).difference(set(cars_in_val_spots))}")

print("\nIntersection between train and val:")

intersection = set(train_cars).intersection(set(val_cars))
OK = True if len(intersection) == 0 else False
print(f"All OK by cars: {OK} - Equal Qty: {len(intersection)} - Equal Cars: {intersection}")

intersection = set(cars_in_train_spots).intersection(set(cars_in_val_spots))
OK = True if len(intersection) == 0 else False
print(f"All OK by spots: {OK} - Equal Qty: {len(intersection)} - Equal Cars: {intersection}")

Train and Val: PLds_CNRParkEXT
Cars qty:
Train # By cars: 3687 - By spots: 3680
Val # By cars: 1941 - By spots: 1931

Difference:
Train: {'31995', '32020', '32003', '31994', '31945', '31993', '31992', '31941', '31991', '31812', '31947', '31998', '31997', '31999', '31814'}
Val: {'31895', '32029', '31891', '32024', '31762', '29137', '30443', '23974', '23231', '31962'}

Intersection between train and val:
All OK by cars: True - Equal Qty: 0 - Equal Cars: set()
All OK by spots: True - Equal Qty: 0 - Equal Cars: set()


##### Train subsets validation

In [5]:
print(f"Train: {config['train'][0].split('/')[-1]}_{config['train'][1].split('/')[-1]}")
train_dataset = ds_loader.get_train_ds()

OK, has_some_different = True, False
for index in range(len(train_dataset)):
    triplet = train_dataset.triplets[index]
    triplet_cars = [car.split("#")[-1].replace(".jpg", "") for car in triplet]
    triplet_spot_ids = [car.split("#")[-2].replace(".jpg", "") for car in triplet]
    
    if full_difficult and len(set(triplet_spot_ids)) != 1:
        OK = False
    elif len(set(triplet_spot_ids)) == 2:
         has_some_different = True 

    if triplet_cars[0] != triplet_cars[1]:
        OK = False

    if triplet_cars[0] == triplet_cars[2]:
        OK = False

print(f"All OK: {OK} - Some different: {has_some_different}")
print(f"Images qty: {len(train_dataset)}")

Train: PLds_CNRParkEXT
All OK: True - Some different: False
Images qty: 3680


##### Val subsets validation

In [6]:
print(f"Train: {config['train'][0].split('/')[-1]}_{config['train'][1].split('/')[-1]}")
val_dataset = ds_loader.get_val_ds()

OK, has_some_different = True, False
for index in range(len(val_dataset)):
    triplet = val_dataset.triplets[index]
    triplet_cars = [car.split("#")[-1].replace(".jpg", "") for car in triplet]
    triplet_spot_ids = [car.split("#")[-2].replace(".jpg", "") for car in triplet]
    
    if full_difficult and len(set(triplet_spot_ids)) != 1:
        OK = False
    elif len(set(triplet_spot_ids)) == 2:
         has_some_different = True 

    if triplet_cars[0] != triplet_cars[1]:
        OK = False

    if triplet_cars[0] == triplet_cars[2]:
        OK = False

print(f"All OK: {OK} - Some different: {has_some_different}")
print(f"Images qty: {len(val_dataset)}")

Train: PLds_CNRParkEXT
All OK: True - Some different: False
Images qty: 3862


##### Test subsets validation

In [7]:
print(f"Test: {config['test'].split('/')[-1]}")

cars_qty, test_pairs, OK = 0, 0, True
test_datasets = ds_loader.get_test_subsets()

for subset, spots in test_sets.items():
    cars_qty += len([car for cars in spots.values() for car in cars])

for name, subset in test_datasets.items():
    test_pairs += len(subset)
    
    for index in range(len(subset)):
        pair, label = subset.images[index], subset.labels[index]
        pair_cars = [car.split("#")[-1].replace(".jpg", "") for car in pair]
        pair_spot_ids = [car.split("#")[-2].replace(".jpg", "") for car in pair]

        if pair_spot_ids[0] != pair_spot_ids[1]:
            OK = False

        if label == 0 and pair_cars[0] != pair_cars[1]:
            OK = False

        if label == 0 and pair[0] == pair[1]:
            OK = False
        
        if label == 1 and pair_cars[0] == pair_cars[1]:
            OK = False

if (cars_qty * 4) != test_pairs:
    OK = False

print(f"All OK: {OK}")

Test: PKLot
All OK: True
